# Supervised Baseline Preparation

## Business question

Can we build a transparent baseline model to prioritize potentially high-risk complaints?

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from cx_complaints.features.text_features import add_text_features
from cx_complaints.modelling.risk import score_complaint_risk
from cx_complaints.modelling.supervised import (
    evaluate_binary_classifier,
    summarize_class_imbalance,
    time_aware_train_test_split,
)
from cx_complaints.preprocessing.text import add_clean_text_column

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected data/raw/cfpb_sample.csv. Run the fetch command first."
    )

df = pd.read_csv(raw_path)
df = add_clean_text_column(df)
df = add_text_features(df)
df = score_complaint_risk(df)
df["is_high_risk"] = (df["risk_level"] == "high").astype(int)
df[["date_received", "clean_text", "risk_level", "is_high_risk"]].head()

In [ ]:
split = time_aware_train_test_split(
    frame=df,
    date_column="date_received",
    train_fraction=0.8,
)

print(f"Train rows: {len(split.train):,}")
print(f"Test rows: {len(split.test):,}")
print(f"Split date: {split.split_timestamp.date()}")

In [ ]:
summarize_class_imbalance(split.train["is_high_risk"])

In [ ]:
vectorizer = TfidfVectorizer(min_df=3, max_features=3000, ngram_range=(1, 2))
x_train = vectorizer.fit_transform(split.train["clean_text"].fillna(""))
x_test = vectorizer.transform(split.test["clean_text"].fillna(""))

y_train = split.train["is_high_risk"]
y_test = split.test["is_high_risk"]

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(x_train, y_train)

pred = pd.Series(model.predict(x_test), index=split.test.index)
score = pd.Series(model.predict_proba(x_test)[:, 1], index=split.test.index)
metrics = evaluate_binary_classifier(y_true=y_test, y_pred=pred, y_score=score)
metrics

In [ ]:
top_features = pd.DataFrame(
    {
        "feature": vectorizer.get_feature_names_out(),
        "coefficient": model.coef_[0],
    }
)

important_positive = top_features.sort_values("coefficient", ascending=False).head(15)
important_negative = top_features.sort_values("coefficient", ascending=True).head(15)

important_positive

In [ ]:
important_negative

## Limitations and next steps

1. This uses heuristic `risk_level` as a proxy label, not validated outcomes.
2. Add calibration and threshold analysis before operational use.
3. Add metadata features and compare against linear SVM baseline.